In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys, pathlib
import numpy as np
import pandas as pd

# Your existing working directory
WORKDIR = "/content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods"


os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

SUPP_DIR = os.path.join(WORKDIR, "Supplement")
if os.path.isdir(SUPP_DIR) and SUPP_DIR not in sys.path:
    sys.path.insert(0, SUPP_DIR)

# Ensure 'Supplement' is a package if only estimation.py exists
init_path = os.path.join(SUPP_DIR, "__init__.py")
if os.path.isdir(SUPP_DIR) and not os.path.exists(init_path):
    with open(init_path, "w", encoding="utf-8") as f:
        f.write("from .estimation import *\n")
    print("[info] Created Supplement/__init__.py shim")

In [3]:
!pip install -q rpy2

%load_ext rpy2.ipython


In [4]:
%%R
if (!requireNamespace("grf", quietly = TRUE)) {
  install.packages("grf", repos = "https://cloud.r-project.org")
}


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘zoo’, ‘DiceKriging’, ‘lmtest’, ‘sandwich’, ‘RcppEigen’

trying URL 'https://cloud.r-project.org/src/contrib/zoo_1.8-15.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/DiceKriging_1.6.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/lmtest_0.9-40.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/sandwich_3.1-1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/RcppEigen_0.3.4.0.2.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/grf_2.6.1.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmp7TEOsH/downloaded_packages’


In [5]:
# -*- coding: utf-8 -*-
"""
Colab script: build semi-synthetic base objects for the Job Corps empirical application (GRF-based).

What this script does:
  1) Load the empirical application dataset (emp_app.csv) and apply the exact same
     preprocessing as the baseline code (fixed shuffle + one-hot for int64 columns).
  2) Fit a GRF model for the regression function f*(x, t) using (t = d, x = covariates).
  3) Create and save a “semi-synthetic base” CSV that contains:
       - mu_hat_grf = f_hat(X_i, T_i) evaluated at observed (X_i, T_i)
       - g_grf      = residuals Y_i - mu_hat_grf
  4) Compute h*(t) = E_X[f*(X, t)] over a fixed t-grid and save it as a separate CSV.

Outputs (saved under EMP_DIR):
  - "semi-syn data grf.csv"
  - "h_star_grf_empapp.csv"
"""

# --- 1) Imports & paths ---
import pathlib
import numpy as np
import pandas as pd

# GRF (R grf wrapper)
from Supplement.rgrf import regression_forest as RF_grf

# Empirical Application directory
EMP_DIR = "/content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results"
current_path = pathlib.Path(EMP_DIR)

# --- 2) Load empirical data + apply baseline preprocessing ---
data_path = current_path / "emp_app.csv"
print(f"Reading {data_path} ...")
data = pd.read_csv(data_path, index_col=0)

# Fixed shuffle
data = data.sample(frac=1, random_state=20)

# One-hot encode int64 columns
data = pd.concat(
    [
        data.select_dtypes(exclude="int64"),
        pd.get_dummies(
            data.select_dtypes("int64").astype("category"),
            drop_first=True,
            dtype=float,
        ),
    ],
    axis=1,
)

# Split into covariates X, treatment T, outcome Y (same as baseline)
X = data.drop(["d", "y"], axis=1)
T = data["d"]
Y = data["y"]

print("Shapes:")
print("  X:", X.shape)
print("  T:", T.shape)
print("  Y:", Y.shape)

# --- 3) Fit GRF regression for f*(x,t) and construct (mu_hat, g) ---
def fit_fhat_semi_grf(X: pd.DataFrame, T: pd.Series, Y: pd.Series):
    """
    Fit a GRF regression model for f*(x,t) using features [a, x].

    Returns
    -------
    grf_model : fitted GRF model object
    mu_hat : np.ndarray
        Predictions f_hat(X_i, T_i) evaluated at observed (X_i, T_i).
    g : np.ndarray
        Residuals g_i = Y_i - mu_hat_i.
    fhat : callable
        Function fhat(x_block, t_scalar) that predicts f_hat(x, t) for:
          - x_block : DataFrame of covariates (n x p)
          - t_scalar: scalar treatment value t
    """
    # Design matrix for GRF: concatenate treatment column and covariates
    X_rf = pd.concat(
        [
            T.rename("d").reset_index(drop=True),
            X.reset_index(drop=True),
        ],
        axis=1,
    )

    # rgrf.py is assumed to expose a sklearn-like API
    grf_model = RF_grf()

    # Fit the GRF model on (t, x) -> y
    grf_model.fit(X_rf.values, Y.values)

    # Predictions at observed (X_i, T_i)
    mu_hat = grf_model.predict(X_rf.values)
    g = Y.values - mu_hat

    # Predictor fhat(x_block, t_scalar) for arbitrary t on a block of x's
    def fhat(x_block: pd.DataFrame, t_scalar: float):
        x_block = x_block.reset_index(drop=True)
        n = len(x_block)
        T_col = np.full((n, 1), float(t_scalar))
        X_pred = np.hstack([T_col, x_block.values])
        return grf_model.predict(X_pred)

    return grf_model, mu_hat, g, fhat


print("Fitting GRF for f*(x,t)...")
grf_model, mu_hat, g, fhat = fit_fhat_semi_grf(X, T, Y)
print("Done. Example mu_hat[0], g[0]:", mu_hat[0], g[0])

# --- 4) Save semi-synthetic base data (mu_hat_grf, g_grf) ---
# Attach mu_hat and g to the processed empirical dataset and save to CSV
out_df = data.copy()
out_df["mu_hat_grf"] = mu_hat
out_df["g_grf"] = g

out_name = "semi-syn data grf.csv"
out_path = current_path / out_name

out_df.to_csv(out_path, index=True)
print(f"Saved semi-synthetic base data (with mu_hat_grf, g_grf) to:\n  {out_path}")

# --- 5) Compute h*(t) = E_X[f*(X,t)] over a fixed t-grid and save ---
# t-grid used in the baseline empirical application
t_list = np.arange(160, 2001, 40)


def compute_h_star_over_grid(X: pd.DataFrame, fhat, t_list: np.ndarray):
    """
    Compute h*(t) = E_X[f*(X,t)] on a given t-grid using the fitted fhat.

    Parameters
    ----------
    X : DataFrame
        Covariates (n x p).
    fhat : callable
        Function fhat(X_block, t_scalar) -> predictions (n,).
    t_list : np.ndarray
        Grid of treatment values.

    Returns
    -------
    np.ndarray
        h_star values aligned with t_list.
    """
    h_vals = []
    for t in t_list:
        vals = fhat(X, t)  # shape (n,)
        h_vals.append(np.mean(vals))
    return np.array(h_vals)


print("Computing h^*(t) over grid ...")
h_star_vals = compute_h_star_over_grid(X, fhat, t_list)

h_df = pd.DataFrame({
    "t": t_list,
    "h_star": h_star_vals,
})

h_out_name = "h_star_grf_empapp.csv"
h_out_path = current_path / h_out_name

h_df.to_csv(h_out_path, index=False)
print(f"Saved h^*(t) to:\n  {h_out_path}")


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1305: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


Reading /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/emp_app.csv ...
Shapes:
  X: (4024, 138)
  T: (4024,)
  Y: (4024,)
Fitting GRF for f*(x,t)...
Done. Example mu_hat[0], g[0]: 31.951727843049817 -31.951727843049817
Saved semi-synthetic base data (with mu_hat_grf, g_grf) to:
  /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/semi-syn data grf.csv
Computing h^*(t) over grid ...
Saved h^*(t) to:
  /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/h_star_grf_empapp.csv
